# Interaction Tracking Analysis

Code to analyze the output from `interactionTracker.js`: process it to determine the likely interactions and reduce the amount of data, then produce some statistics.

In [1]:
from helpers import *


participants = [
    EvaluationData("P1", TaskOrder.T1T2, EvaluationTask.T1),
    EvaluationData("P2", TaskOrder.T1T2, EvaluationTask.T2),
    EvaluationData("P3", TaskOrder.T2T1, EvaluationTask.T2),
    EvaluationData("P4", TaskOrder.T1T2, EvaluationTask.T1),
    EvaluationData("P5", TaskOrder.T1T2, EvaluationTask.T2),
    EvaluationData("P6", TaskOrder.T2T1, EvaluationTask.T1),
    EvaluationData("P7", TaskOrder.T2T1, EvaluationTask.T1),
    EvaluationData("P8", TaskOrder.T2T1, EvaluationTask.T2),
    EvaluationData("P9", TaskOrder.T1T2, EvaluationTask.T2)
]

for participant in participants:
    filepath = os.getcwd() + f"/data/interactions_{participant.name}.json"
    interactionData = loadInteractionData(filepath)
    participant.interactionData = interactionData

In [2]:
# Load (refined) interaction data.
for participant in participants:
    print("\nParticipant " + participant.name)
    filepath = os.getcwd() + f"/data/interactions_{participant.name}.json"
    interactionData = loadInteractionData(filepath)
    participant.interactionData = refineInteractionData(interactionData)


Participant P1
Removed 37 invalid interaction entries!
Removed 12 duplicate interaction entries.

Participant P2
Removed 58 invalid interaction entries!

Participant P3
Removed 99 invalid interaction entries!
Removed 44 duplicate interaction entries.

Participant P4
Removed 21 invalid interaction entries!

Participant P5
Removed 165 invalid interaction entries!

Participant P6
Removed 117 invalid interaction entries!
Removed 42 duplicate interaction entries.

Participant P7
Removed 56 invalid interaction entries!

Participant P8
Removed 168 invalid interaction entries!
Removed 2 duplicate interaction entries.

Participant P9
Removed 25 invalid interaction entries!
Removed 3 duplicate interaction entries.


In [3]:
# Map interactions to correct tasks.
for participant in participants:
    interactionsFirstTask, interactionsSecondTask = splitInteractionsByTask(participant.interactionData)

    # only data for one task was tracked -> assume it was for task with extension active
    if not interactionsSecondTask:
        if participant.evalTask == EvaluationTask.T1:
            participant.interactionsTaskOne = interactionsFirstTask
            participant.interactionsTaskTwo = []
        else:
            participant.interactionsTaskOne = []
            participant.interactionsTaskTwo = interactionsFirstTask
    
    # ensure interactions are mapped to correct task
    else:
        if participant.taskOrder == TaskOrder.T1T2:
            participant.interactionsTaskOne = interactionsFirstTask
            participant.interactionsTaskTwo = interactionsSecondTask
        else:
            participant.interactionsTaskOne = interactionsSecondTask
            participant.interactionsTaskTwo = interactionsFirstTask

In [4]:
taskOneWithExtensionData = []
taskOneWithoutExtensionData = []
taskTwoWithExtensionData = []
taskTwoWithoutExtensionData = []

for participant in participants:
    if participant.interactionsTaskOne:
        if participant.evalTask == EvaluationTask.T1:
            taskOneWithExtensionData.append(countInteractions(participant.interactionsTaskOne))
        else:
            taskOneWithoutExtensionData.append(countInteractions(participant.interactionsTaskOne))
    if participant.interactionsTaskTwo:
        if participant.evalTask == EvaluationTask.T2:
            taskTwoWithExtensionData.append(countInteractions(participant.interactionsTaskTwo))
        else:
            taskTwoWithoutExtensionData.append(countInteractions(participant.interactionsTaskTwo))

def combineInteractions(interactionsData):
    combinedInteractions = defaultdict(int)
    for participantData in interactionsData:
        for key, value in participantData.items():
            combinedInteractions[key] += value
    return combinedInteractions


taskOneWithExtension = combineInteractions(taskOneWithExtensionData)
taskOneWithoutExtension = combineInteractions(taskOneWithoutExtensionData)
taskTwoWithExtension = combineInteractions(taskTwoWithExtensionData)
taskTwoWithoutExtension = combineInteractions(taskTwoWithoutExtensionData)

print(taskOneWithExtension)
print(taskOneWithoutExtension)
print(taskTwoWithExtension)
print(taskTwoWithoutExtension)

defaultdict(<class 'int'>, {'UnknownJump': 20, 'Scroll': 576, 'ChangeVisibleRanges': 116, 'ChangeFile': 56, 'ChangeSidebarVisibility': 10, 'EditingSession': 136, 'EditFile': 39, 'NavigationJump': 38, 'SwitchView': 2})
defaultdict(<class 'int'>, {'ChangeSidebarVisibility': 3, 'ChangeFile': 70, 'ChangeVisibleRanges': 115, 'Scroll': 235, 'EditingSession': 45, 'EditFile': 24, 'UnknownJump': 11, 'NavigationJump': 21})
defaultdict(<class 'int'>, {'ChangeSidebarVisibility': 9, 'ChangeFile': 180, 'Scroll': 534, 'ChangeVisibleRanges': 410, 'UnknownJump': 1893, 'EditingSession': 157, 'EditFile': 66, 'NavigationJump': 10, 'SwitchView': 1})
defaultdict(<class 'int'>, {'ChangeSidebarVisibility': 3, 'Scroll': 419, 'ChangeFile': 40, 'UnknownJump': 57, 'ChangeVisibleRanges': 191, 'EditingSession': 74, 'EditFile': 31})


In [5]:
for participant in participants:
    print("\n--- Participant " + participant.name + " ---")

    if participant.interactionsTaskOne:
        print("\nTask One")
        print(countInteractions(participant.interactionsTaskOne))
        print(f"Scrolling Distance: {getScrollingDistance(participant.interactionsTaskOne)}")
        print(f"Duration: {(participant.interactionsTaskOne[-1]['timeStamp'] - participant.interactionsTaskOne[0]['timeStamp']) / 1000}")

    if participant.interactionsTaskTwo:
        print("\nTask Two")
        print(countInteractions(participant.interactionsTaskTwo))
        print(f"Scrolling Distance: {getScrollingDistance(participant.interactionsTaskTwo)}")
        print(f"Duration: {(participant.interactionsTaskTwo[-1]['timeStamp'] - participant.interactionsTaskTwo[0]['timeStamp']) / 1000}")


--- Participant P1 ---

Task One
defaultdict(<class 'int'>, {'UnknownJump': 10, 'Scroll': 148, 'ChangeVisibleRanges': 32, 'ChangeFile': 21, 'ChangeSidebarVisibility': 1, 'EditingSession': 57, 'EditFile': 11, 'NavigationJump': 8})
Scrolling Distance: 2194
Duration: 1648.144

--- Participant P2 ---

Task Two
defaultdict(<class 'int'>, {'ChangeSidebarVisibility': 2, 'ChangeFile': 27, 'Scroll': 52, 'ChangeVisibleRanges': 43, 'UnknownJump': 17, 'EditingSession': 38, 'EditFile': 16, 'NavigationJump': 2})
Scrolling Distance: 744
Duration: 1463.525

--- Participant P3 ---

Task Two
defaultdict(<class 'int'>, {'EditFile': 18, 'EditingSession': 54, 'ChangeFile': 47, 'ChangeVisibleRanges': 226, 'Scroll': 233, 'UnknownJump': 1822, 'ChangeSidebarVisibility': 3, 'NavigationJump': 4})
Scrolling Distance: 4808
Duration: 1611.695

--- Participant P4 ---

Task One
defaultdict(<class 'int'>, {'ChangeSidebarVisibility': 1, 'ChangeFile': 5, 'ChangeVisibleRanges': 11, 'Scroll': 133, 'NavigationJump': 18, '